In [1]:
# ✅ Spam Detection using BERT (Complete Single Cell)

# ----------------------------
# 🧩 Setup
# ----------------------------
!pip install -q transformers datasets torch scikit-learn

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
import pandas as pd
import numpy as np

# ----------------------------
# 📦 Load Dataset (SMS Spam Collection)
# ----------------------------
# Download the dataset from Hugging Face
dataset = load_dataset("sms_spam")

# Convert to train/test
train_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_data = train_dataset["train"]
test_data = train_dataset["test"]

# ----------------------------
# 🧹 Preprocessing
# ----------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def clean_text(text):
    import re
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def tokenize_function(examples):
    return tokenizer([clean_text(t) for t in examples["sms"]], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ----------------------------
# 🤖 Model Definition
# ----------------------------
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# ----------------------------
# ⚙️ Training Arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="no",
    num_train_epochs=1,  # small for demo; increase for better accuracy
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=10,
)

# ----------------------------
# 🧮 Metrics
# ----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    return {"accuracy": acc, "f1": f1}

# ----------------------------
# 🚀 Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# ----------------------------
# 🏋️ Train
# ----------------------------
trainer.train()

# ----------------------------
# 📊 Evaluation
# ----------------------------
predictions = trainer.predict(test_data)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("\n✅ Classification Report:\n", classification_report(y_true, y_pred))
print("\n✅ Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ----------------------------
# 💬 Demo Predictions
# ----------------------------
test_msgs = [
    "Congratulations! You won a prize!",
    "Please review the report and send feedback.",
    "Free entry in 2 a weekly competition to win FA Cup final tickets!",
    "Can we meet tomorrow morning?"
]

inputs = tokenizer(test_msgs, return_tensors="pt", padding=True, truncation=True, max_length=128)
outputs = model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)

print("\n💡 Demo Results:")
for msg, pred in zip(test_msgs, preds):
    print(f"Message: {msg}\nPrediction: {'Spam' if pred==1 else 'Not Spam'}\n")

# ----------------------------
# 🧾 Log Results (Simulated Trello Comment)
# ----------------------------
print("📌 Logged Results:")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_true, y_pred):.4f}")
print("✅ Task Completed: Model trained, evaluated, and demo tested.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [2]:
# ✅ Spam Detection using BERT (Fully Working Single-Cell Code for Colab)
# Covers: Setup → Dataset → Preprocessing → Model → Training → Evaluation → Demo → Log

!pip install -q transformers==4.31.0 datasets==2.14.4 torch scikit-learn

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np
import torch
import re

# ----------------------------
# 🧩 Setup + Dataset
# ----------------------------
dataset = load_dataset("sms_spam")

# Split into train/test
train_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_data, test_data = train_dataset["train"], train_dataset["test"]

# ----------------------------
# 🧹 Preprocessing
# ----------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def tokenize_function(examples):
    cleaned = [clean_text(t) for t in examples["sms"]]
    return tokenizer(cleaned, padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ----------------------------
# 🤖 Model
# ----------------------------
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# ----------------------------
# ⚙️ Training Arguments (compatible version)
# ----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,  # increase to 2–3 for better accuracy
    evaluation_strategy="steps"
)

# ----------------------------
# 🧮 Metrics
# ----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

# ----------------------------
# 🚀 Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ----------------------------
# 🏋️ Train
# ----------------------------
trainer.train()

# ----------------------------
# 📊 Evaluation
# ----------------------------
predictions = trainer.predict(test_data)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("\n✅ Classification Report:\n", classification_report(y_true, y_pred))
print("\n✅ Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ----------------------------
# 💬 Demo Predictions
# ----------------------------
sample_msgs = [
    "Congratulations! You have won a free iPhone!",
    "Hey, are we still meeting at 3 PM?",
    "URGENT! You won a $1000 Walmart gift card. Reply YES to claim.",
    "Please review the project report before tomorrow."
]

inputs = tokenizer(sample_msgs, return_tensors="pt", padding=True, truncation=True, max_length=128)
outputs = model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)

print("\n💡 Demo Results:")
for msg, pred in zip(sample_msgs, preds):
    print(f"Message: {msg}\nPrediction: {'Spam' if pred==1 else 'Not Spam'}\n")

# ----------------------------
# 🧾 Log Results (Simulated Trello Comment)
# ----------------------------
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print("📌 Logged Results:")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print("✅ Task Completed: BERT Spam Classifier fine-tuned, evaluated, and tested successfully.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 137.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 13.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based proj

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [3]:
# Single Colab cell: Spam Detection using BERT (manual training loop)
# Works even if tokenizers build failed and avoids Trainer/TrainingArguments API mismatches.

# Optional: try to install recommended versions quietly. If install fails for tokenizers, code still runs.
!pip install -q transformers==4.31.0 datasets==2.14.4 torch scikit-learn || true

import re
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm
import random
import os

# -------- config --------
NUM_EPOCHS = 1         # set to 2-3 for better accuracy
BATCH_SIZE = 8
MAX_LEN = 128
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("Device:", DEVICE)

# -------- load dataset --------
print("Loading SMS Spam dataset...")
ds = load_dataset("sms_spam")["train"]              # dataset has 'sms' and 'label'
# quick train/test split
split = ds.train_test_split(test_size=0.2, seed=SEED)
train_ds = split["train"]
test_ds  = split["test"]
print(f"Train size: {len(train_ds)}, Test size: {len(test_ds)}")

# -------- tokenizer (pure python fallback works if fast tokenizers failed) --------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # pure python tokenizer included in transformers

def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', str(text))
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

# -------- PyTorch dataset wrapper --------
class SpamDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=128):
        self.examples = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        item = self.examples[idx]
        text = clean_text(item["sms"])
        enc = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_dataset = SpamDataset(train_ds, tokenizer, MAX_LEN)
test_dataset  = SpamDataset(test_ds, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
eval_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -------- model --------
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(DEVICE)

# -------- optimizer --------
optimizer = AdamW(model.parameters(), lr=LR)

# -------- training loop --------
model.train()
for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    epoch_losses = []
    prog = tqdm(train_loader, desc="Training", leave=False)
    for batch in prog:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_losses.append(loss.item())
        prog.set_postfix(loss=np.mean(epoch_losses))
    print(f"Epoch {epoch+1} avg loss: {np.mean(epoch_losses):.4f}")

# -------- evaluation --------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
print("\n✅ Evaluation Results")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:\n", classification_report(all_labels, all_preds, target_names=["Not Spam","Spam"]))
print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

# -------- demo predictions --------
def predict_texts(texts):
    model.eval()
    texts_clean = [clean_text(t) for t in texts]
    enc = tokenizer(texts_clean, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(out.logits, dim=1).cpu().numpy().tolist()
    return preds

sample_msgs = [
    "Congratulations! You have won a free iPhone!",
    "Hey, are we still meeting at 3 PM?",
    "URGENT! You won a $1000 Walmart gift card. Reply YES to claim.",
    "Please review the project report before tomorrow."
]

preds = predict_texts(sample_msgs)
print("\n💡 Demo Results:")
for m, p in zip(sample_msgs, preds):
    print(f"Message: {m}\nPrediction: {'Spam' if p==1 else 'Not Spam'}\n")

# -------- save model locally (optional) --------
OUT_DIR = "/content/spam_bert_saved"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Saving model to {OUT_DIR} ...")
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Saved. You can zip/download or copy to Drive.")

# -------- simulated Trello log --------
print("\n📌 Trello Log (simulated):")
print(f"- Model fine-tuned for {NUM_EPOCHS} epoch(s). Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")
print(f"- Accuracy: {acc:.4f}, F1: {f1:.4f}")
print("- Confusion matrix and classification report printed above.")
print("- Demo messages tested.")
print("- Saved model to /content/spam_bert_saved for deployment.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [4]:
# ✅ Spam Detection using BERT (Colab-Safe, Single-Cell, Python 3.12 Compatible)

!pip install -q transformers torch datasets scikit-learn tqdm

import re
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# -----------------------------
# ⚙️ Configuration
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
MAX_LEN = 128
EPOCHS = 1           # increase to 2–3 for better accuracy
LR = 2e-5
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

print("Device:", DEVICE)

# -----------------------------
# 📦 Load Dataset
# -----------------------------
print("Loading dataset...")
ds = load_dataset("sms_spam")["train"]
split = ds.train_test_split(test_size=0.2, seed=SEED)
train_ds, test_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")

# -----------------------------
# 🧹 Preprocessing & Tokenization
# -----------------------------
def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", str(text))
    text = re.sub(r"[^A-Za-z0-9 ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", use_fast=False)

class SpamDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        enc = tokenizer.encode_plus(
            clean_text(item["sms"]),
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_loader = DataLoader(SpamDataset(train_ds), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(SpamDataset(test_ds),  batch_size=BATCH_SIZE)

# -----------------------------
# 🤖 Model & Optimizer
# -----------------------------
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)

# -----------------------------
# 🏋️ Training Loop
# -----------------------------
model.train()
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    total_loss = 0
    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attn = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        out = model(input_ids, attention_mask=attn, labels=labels)
        loss = out.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Average loss: {total_loss/len(train_loader):.4f}")

# -----------------------------
# 📊 Evaluation
# -----------------------------
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(DEVICE)
        attn = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = model(input_ids, attention_mask=attn)
        preds = torch.argmax(outputs.logits, dim=1)
        y_true += labels.cpu().tolist()
        y_pred += preds.cpu().tolist()

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred)
print(f"\n✅ Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["Not Spam","Spam"]))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# -----------------------------
# 💬 Demo Predictions
# -----------------------------
samples = [
    "Congratulations! You won a free vacation trip!",
    "Hey, are we still meeting at 5 PM?",
    "URGENT: Claim your $1000 Walmart gift card now!",
    "Please send the updated project document."
]
enc = tokenizer(samples, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    preds = torch.argmax(model(**enc).logits, dim=1).cpu().tolist()

print("\n💡 Demo Results:")
for msg, p in zip(samples, preds):
    print(f"Message: {msg}\nPrediction: {'Spam' if p==1 else 'Not Spam'}\n")

# -----------------------------
# 💾 Save Model
# -----------------------------
out_dir = "/content/spam_bert_model"
os.makedirs(out_dir, exist_ok=True)
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print(f"Model saved to {out_dir}")

# -----------------------------
# 🧾 Trello-style Log
# -----------------------------
print("\n📌 Trello Log:")
print(f"- Training complete for {EPOCHS} epoch(s)")
print(f"- Accuracy: {acc:.4f}, F1: {f1:.4f}")
print("- Model + tokenizer saved to /content/spam_bert_model")
print("- Demo messages tested successfully ✅")


Device: cpu
Loading dataset...
Train: 4459, Test: 1115


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1/1


Training:   0%|          | 0/558 [00:00<?, ?it/s]

Average loss: 0.0716


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]


✅ Accuracy: 0.9892, F1 Score: 0.9583

Classification Report:
               precision    recall  f1-score   support

    Not Spam       0.99      1.00      0.99       966
        Spam       0.99      0.93      0.96       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.98      1115
weighted avg       0.99      0.99      0.99      1115

Confusion Matrix:
 [[965   1]
 [ 11 138]]

💡 Demo Results:
Message: Congratulations! You won a free vacation trip!
Prediction: Not Spam

Message: Hey, are we still meeting at 5 PM?
Prediction: Not Spam

Message: URGENT: Claim your $1000 Walmart gift card now!
Prediction: Spam

Message: Please send the updated project document.
Prediction: Not Spam

Model saved to /content/spam_bert_model

📌 Trello Log:
- Training complete for 1 epoch(s)
- Accuracy: 0.9892, F1: 0.9583
- Model + tokenizer saved to /content/spam_bert_model
- Demo messages tested successfully ✅
